In [1]:
import pandas as pd
import os
import numpy as np
from datetime import datetime, timedelta
import re

In [2]:
pd.set_option('display.max_columns', None)

#### <span style="color:#FF6347;">**READ**</span> 

In [3]:
# file_path = "P:/Dataset/MO-DBT-data-curation/Cancer/mo_cancer.xlsx"
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "cancer_candidate_cohort" + ".xlsx")
cancer_cohort = pd.read_excel(file_path)
cancer = cancer_cohort[cancer_cohort["Index"]=="INDEX"]
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "normal_candidate" + ".xlsx")
normal_candidate = pd.read_excel(file_path)

file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "cancer_candidate_ehr_unique_patient" + ".xlsx")
cancer_ehr = pd.read_excel(file_path)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "normal_candidate_ehr" + ".xlsx")
normal_candidate_ehr = pd.read_excel(file_path)

In [4]:
normal_candidate_ehr = normal_candidate_ehr[normal_candidate_ehr["Index"]=="INDEX"]

In [5]:
normal_candidate_ehr = normal_candidate_ehr.drop_duplicates(subset=["PATIENT_STUDY_ID"], keep="first")

In [6]:
cancer_ehr.shape, cancer_ehr["PATIENT_STUDY_ID"].nunique(), normal_candidate_ehr.shape, normal_candidate_ehr["PATIENT_STUDY_ID"].nunique()

((1630, 63), 1630, (30787, 62), 30787)

# **Step 1. BI-RADS**

In [7]:
cancer["FINDING_CATEGORY"].unique(), normal_candidate["FINDING_CATEGORY"].unique()

(array(['5 - Highly suggestive of malignancy, appropriate action should be taken',
        '2 - Benign finding',
        '4 - Suspicious abnormality, biopsy should be considered',
        '0 - Need additional imaging evaluation',
        '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion',
        '99 - Post procedure mammograms for marker placement',
        '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion',
        '3 - Probably benign - short interval follow-up', '1 - Negative',
        '4A - Suspicious abnormality - biopsy should be considered - low suspicion',
        '6 - Known biopsy proven malignancy'], dtype=object),
 array(['2 - Benign finding', '1 - Negative'], dtype=object))

In [8]:
cancer["COMPOSITION_NAME"].unique(), normal_candidate["COMPOSITION_NAME"].unique()

(array(['Heterogeneously dense (51% - 75%)', 'Extremely dense (>75%)',
        'Scattered fibroglandular (25% - 50%)', nan], dtype=object),
 array(['Heterogeneously dense (51% - 75%)',
        'Scattered fibroglandular (25% - 50%)', 'Extremely dense (>75%)',
        'Almost entirely fat (<25%)'], dtype=object))

In [9]:
# ── STEP 1: Bilateral BIRADS combo + Density at INDEX-1 ───────────────────────
BIRADS_SHORT = {
    '0 - Need additional imaging evaluation': '0',
    '1 - Negative': '1', 
    '2 - Benign finding': '2',
    '3 - Probably benign - short interval follow-up': '3',
    '4 - Suspicious abnormality, biopsy should be considered': '4',
    '4A - Suspicious abnormality - biopsy should be considered - low suspicion': '4',
    '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion': '4',
    '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion': '4',
    '5 - Highly suggestive of malignancy, appropriate action should be taken': '5',
    '6 - Known biopsy proven malignancy': '6'
    }

DENSITY_SHORT = {
    'Almost entirely fat (<25%)': 'A',
    'Scattered fibroglandular (25% - 50%)': 'B',
    'Heterogeneously dense (51% - 75%)': 'C',
    'Extremely dense (>75%)': 'D'
}

c1 = cancer[cancer['Index'] == 'INDEX'].copy()
n1 = normal_candidate[normal_candidate['Index'] == 'INDEX'].copy()

def get_bilateral_combo(df):
    df = df.copy()
    df['CAT'] = df['FINDING_CATEGORY'].map(BIRADS_SHORT)

    side_dedup = df.drop_duplicates(subset=['PATIENT_STUDY_ID', 'SIDE'], keep='first')
    valid = side_dedup[side_dedup['SIDE'].isin(['L', 'R'])]
    piv = valid.pivot_table(
        index='PATIENT_STUDY_ID', columns='SIDE', values='CAT', aggfunc='first'
    )
    piv.columns.name = None
    def combo(row):
        vals = sorted([v for v in row.dropna()])
        return '+'.join(vals) if vals else None
    piv['birads_combo'] = piv.apply(combo, axis=1)
    result = piv[['birads_combo']].reset_index()

    nan_side_pids = set(df['PATIENT_STUDY_ID']) - set(result['PATIENT_STUDY_ID'])
    if nan_side_pids:
        nan_side = (
            df[df['PATIENT_STUDY_ID'].isin(nan_side_pids)]
            .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
            [['PATIENT_STUDY_ID', 'CAT']]
            .rename(columns={'CAT': 'birads_combo'})
        )
        result = pd.concat([result, nan_side], ignore_index=True)

    # ── Add density (one value per patient; take first non-null) ──────────────
    density = (
        df.dropna(subset=['COMPOSITION_NAME'])          # ← adjust col name if needed
        .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
        [['PATIENT_STUDY_ID', 'COMPOSITION_NAME']]
    )
    density['density'] = density['COMPOSITION_NAME'].map(DENSITY_SHORT)
    result = result.merge(density[['PATIENT_STUDY_ID', 'density']], on='PATIENT_STUDY_ID', how='left')

    return result

def combo_table(df, label, var):
    n_total = df['PATIENT_STUDY_ID'].nunique()
    counts  = df[var].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'Combo': counts.index,
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*45}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*45}")
    print(tbl.to_string(index=False))
    print()

In [10]:
c_combo = get_bilateral_combo(c1)
n_combo = get_bilateral_combo(n1)

In [11]:
combo_table(c_combo, 'Cancer', 'birads_combo')
combo_table(n_combo, 'Normal Candidate', 'birads_combo')

─────────────────────────────────────────────
Cancer  (n=1,630)
─────────────────────────────────────────────
Combo  Count    %
    4    602 36.9
  2+4    256 15.7
  4+4    157  9.6
  2+5    152  9.3
  0+0    151  9.3
    5    150  9.2
  5+5     68  4.2
  2+2     28  1.7
  1+4     23  1.4
  1+5     17  1.0
  3+4      7  0.4
  3+5      6  0.4
  4+5      5  0.3
  1+1      4  0.2
  0+5      2  0.1
  3+3      1  0.1
  0+4      1  0.1

─────────────────────────────────────────────
Normal Candidate  (n=30,789)
─────────────────────────────────────────────
Combo  Count    %
  2+2  19953 64.8
  1+1   9675 31.4
    1    950  3.1
    2    211  0.7



In [12]:
combo_table(c_combo, 'Cancer', 'density')
combo_table(n_combo, 'Normal Candidate', 'density')

─────────────────────────────────────────────
Cancer  (n=1,630)
─────────────────────────────────────────────
Combo  Count    %
    C   1538 94.4
    B     54  3.3
    D     37  2.3
  NaN      1  0.1

─────────────────────────────────────────────
Normal Candidate  (n=30,789)
─────────────────────────────────────────────
Combo  Count    %
    C  27616 89.7
    B   2453  8.0
    D    715  2.3
    A      5  0.0



# **Step 2. Race**

In [13]:
c_race_cols = [c for c in cancer_ehr.columns if c.startswith('RACE_')]
n_race_cols = [c for c in normal_candidate_ehr.columns if c.startswith('RACE_')]

In [14]:
ASIAN_GROUPS = {
    'KOREAN', 'OTHER ASIAN', 'VIETNAMESE', 'CHINESE', 'FILIPINO',
    'JAPANESE', 'INDIAN (ASIAN)'
}

PACIFIC_GROUPS = {
    'HAWAIIAN',  'OTHER PACIFIC ISLANDER'
}

def get_simplified_race(df):
    race_cols = [c for c in df.columns if c.startswith('RACE_')]
    df2 = df[race_cols].fillna(0)
    all_zero = df2.sum(axis=1) == 0
    race = df2.idxmax(axis=1).str.replace('RACE_', '')
    race[all_zero] = 'UNKNOWN'
    def simplify(r):
        if r == 'WHITE':  return 'WHITE'
        if r == 'BLACK':  return 'BLACK'
        if r == 'AMERICAN INDIAN/ALASKA NATIVE':  return 'AMERICAN INDIAN/ALASKA NATIVE'
        if r in PACIFIC_GROUPS: return 'HAWAIIAN/PACIFIC ISLANDER'
        if r in ASIAN_GROUPS: return 'ASIAN'
        return 'OTHER/UNKNOWN'
    return race.apply(simplify)

cancer_ehr['race']        = get_simplified_race(cancer_ehr)
normal_candidate_ehr['race'] = get_simplified_race(normal_candidate_ehr)

In [15]:
def race_summary(df, label):
    n_total = len(df)
    counts = df['race'].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'Race':  counts.index,
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*40}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*40}")
    print(tbl.to_string(index=False))
    print()

race_summary(cancer_ehr,        'Cancer')
race_summary(normal_candidate_ehr, 'Normal Candidate')

────────────────────────────────────────
Cancer  (n=1,630)
────────────────────────────────────────
                         Race  Count    %
                        WHITE   1463 89.8
                        BLACK     94  5.8
                        ASIAN     40  2.5
                OTHER/UNKNOWN     29  1.8
AMERICAN INDIAN/ALASKA NATIVE      4  0.2

────────────────────────────────────────
Normal Candidate  (n=30,787)
────────────────────────────────────────
                         Race  Count    %
                        WHITE  27830 90.4
                        BLACK   1502  4.9
                        ASIAN    757  2.5
                OTHER/UNKNOWN    662  2.2
AMERICAN INDIAN/ALASKA NATIVE     24  0.1
    HAWAIIAN/PACIFIC ISLANDER     12  0.0



In [16]:
cancer_ehr['PATIENT_STUDY_ID'].nunique(), normal_candidate_ehr['PATIENT_STUDY_ID'].nunique()

(1630, 30787)

# **Step 3. Age**

In [17]:
cancer_ehr['PatientAge'].min(), cancer_ehr['PatientAge'].max()

(26.0, 92.0)

In [18]:
AGE_BINS   = list(range(40, 95, 5))
AGE_LABELS = [f'{b}-{b+4}' for b in AGE_BINS[:-1]]

cancer_ehr['age_bin']        = pd.cut(cancer_ehr['PatientAge'],        bins=AGE_BINS, labels=AGE_LABELS, right=False)
normal_candidate_ehr['age_bin'] = pd.cut(normal_candidate_ehr['PatientAge'], bins=AGE_BINS, labels=AGE_LABELS, right=False)

In [19]:
def age_summary(df, label):
    n_total = len(df)
    counts = df['age_bin'].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'AGE_LABELS':  counts.index,
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*40}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*40}")
    print(tbl.to_string(index=False))
    print()

age_summary(cancer_ehr,        'Cancer')
age_summary(normal_candidate_ehr, 'Normal Candidate')

────────────────────────────────────────
Cancer  (n=1,630)
────────────────────────────────────────
AGE_LABELS  Count    %
     65-69    246 15.1
     60-64    235 14.4
     55-59    220 13.5
     70-74    209 12.8
     50-54    171 10.5
     75-79    142  8.7
     45-49    141  8.7
     40-44    113  6.9
       NaN     71  4.4
     80-84     64  3.9
     85-89     18  1.1

────────────────────────────────────────
Normal Candidate  (n=30,787)
────────────────────────────────────────
AGE_LABELS  Count    %
     60-64   4856 15.8
     65-69   4489 14.6
     55-59   4217 13.7
     50-54   3951 12.8
     70-74   3808 12.4
     45-49   3379 11.0
     75-79   2311  7.5
     40-44   1819  5.9
     80-84   1167  3.8
       NaN    404  1.3
     85-89    386  1.3



# **Step 4. Match**

In [20]:
# ── Config ────────────────────────────────────────────────────────────────────
RANDOM_SEED = 42
N_MATCH     = 4  # normals per cancer patient

In [21]:
# ── Feature tables ────────────────────────────────────────────────────
c_feat = c_combo.merge(cancer_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')
n_feat = n_combo.merge(normal_candidate_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')

def make_stratum(df):
    return df['density'].astype(str) + '|' + df['race'].astype(str) + '|' + df['age_bin'].astype(str)

c_feat['stratum'] = make_stratum(c_feat)
n_feat['stratum'] = make_stratum(n_feat)

# ── Filter out unresolvable cancer entries before matching ────────────────────
c_feat_match = c_feat[
    c_feat['age_bin'].notna() &
    (c_feat['race'] != 'OTHER/UNKNOWN') &
    (c_feat['density'].isin(['C', 'D']))
].copy()

print(f"Cancer patients used for matching : {len(c_feat_match)} / {len(c_feat)}")
print(f"Excluded (NaN age or unknown race): {len(c_feat) - len(c_feat_match)}")

Cancer patients used for matching : 1486 / 1630
Excluded (NaN age or unknown race): 144


In [22]:
c_feat_match

,PATIENT_STUDY_ID,birads_combo,density,PatientAge,age_bin,race,stratum
0,4330018595,5,C,54.0,50-54,ASIAN,C|ASIAN|50-54
2,4330303540,0+0,C,79.0,75-79,WHITE,C|WHITE|75-79
5,4330343472,2+5,C,44.0,40-44,WHITE,C|WHITE|40-44
6,4330344296,4,C,66.0,65-69,WHITE,C|WHITE|65-69
8,4330358672,2+5,C,50.0,50-54,WHITE,C|WHITE|50-54
...,...,...,...,...,...,...,...
1625,4339030625,2+4,C,48.0,45-49,WHITE,C|WHITE|45-49
1626,4339047841,4+4,C,57.0,55-59,WHITE,C|WHITE|55-59
1627,4339077225,2+5,C,47.0,45-49,WHITE,C|WHITE|45-49
1628,4339604670,5,C,66.0,65-69,WHITE,C|WHITE|65-69


In [23]:
c_feat_match['density'].unique(), c_feat_match['age_bin'].unique(), c_feat_match['race'].unique()

(array(['C', 'D'], dtype=object),
 ['50-54', '75-79', '40-44', '65-69', '45-49', '80-84', '70-74', '60-64', '55-59', '85-89']
 Categories (10, object): ['40-44' < '45-49' < '50-54' < '55-59' ... '70-74' < '75-79' < '80-84' < '85-89'],
 array(['ASIAN', 'WHITE', 'BLACK', 'AMERICAN INDIAN/ALASKA NATIVE'],
       dtype=object))

In [24]:
# ── Match ─────────────────────────────────────────────────────────────
rng = np.random.default_rng(RANDOM_SEED)
normal_pool = n_feat.copy()
# Filter normal pool to bilateral-only (1+1 or 2+2)
normal_pool = n_feat[n_feat['birads_combo'].isin(['1+1', '2+2'])].copy()
print(f"Normal pool after bilateral filter: {len(normal_pool)} / {len(n_feat)}")

normal_pool_by_stratum = {s: grp.index.tolist() for s, grp in normal_pool.groupby('stratum')}

matched_cancer_rows, matched_normal_rows, unmatched_cancer_ids = [], [], []
used_normal_indices = set()

for _, cancer_row in c_feat_match.iterrows():
    stratum   = cancer_row['stratum']
    available = [i for i in normal_pool_by_stratum.get(stratum, []) if i not in used_normal_indices]

    # Fallback: relax age bin ±5 then ±10 years if stratum is empty
    if len(available) == 0:
        density, race, age_bin = stratum.split('|')
        age_idx = AGE_LABELS.index(age_bin) if age_bin in AGE_LABELS else -1
        for offset in [-1, 1, -2, 2]:
            adj_idx = age_idx + offset
            if 0 <= adj_idx < len(AGE_LABELS):
                adj_stratum = f'{density}|{race}|{AGE_LABELS[adj_idx]}'
                available += [i for i in normal_pool_by_stratum.get(adj_stratum, []) if i not in used_normal_indices]

    if len(available) == 0:
        unmatched_cancer_ids.append(cancer_row['PATIENT_STUDY_ID'])
        continue

    chosen_idx = rng.choice(available, size=min(N_MATCH, len(available)), replace=False).tolist()
    used_normal_indices.update(chosen_idx)
    matched_cancer_rows.append(cancer_row)
    for idx in chosen_idx:
        row = normal_pool.loc[idx].copy()
        row['matched_cancer_id'] = cancer_row['PATIENT_STUDY_ID']
        matched_normal_rows.append(row)

matched_cancer_list  = pd.DataFrame(matched_cancer_rows).reset_index(drop=True)
matched_normal_list = pd.DataFrame(matched_normal_rows).reset_index(drop=True)

# ── Summary ───────────────────────────────────────────────────────────
print(f"Cancer matched    : {len(matched_cancer_list)} / {len(c_feat_match)}")
print(f"Normals selected  : {len(matched_normal_list)}  (ratio {len(matched_normal_list)/len(matched_cancer_list):.2f}:1)")
print(f"Unmatched IDs     : {unmatched_cancer_ids}")

Normal pool after bilateral filter: 29628 / 30789
Cancer matched    : 1486 / 1486
Normals selected  : 5933  (ratio 3.99:1)
Unmatched IDs     : []


In [25]:
# ── Side-by-side distribution summary: Cancer vs Matched Normals ──────────────

def dist_summary(df, label):
    n = len(df)
    print(f"\n{'='*50}")
    print(f"{label}  (n={n})")
    print(f"{'='*50}")

    print("\n── BIRADS Combo ──────────────────────────────")
    for k, v in df['birads_combo'].value_counts(dropna=False).items():
        print(f"  {str(k):15s}: {v:4d}  ({v/n*100:.1f}%)")

    print("\n── Density ───────────────────────────────────")
    for k, v in df['density'].value_counts(dropna=False).items():
        print(f"  {str(k):15s}: {v:4d}  ({v/n*100:.1f}%)")

    print("\n── Age Bin ───────────────────────────────────")
    for k, v in df['age_bin'].value_counts(dropna=False).sort_index().items():
        print(f"  {str(k):15s}: {v:4d}  ({v/n*100:.1f}%)")

    print("\n── Race ──────────────────────────────────────")
    for k, v in df['race'].value_counts(dropna=False).items():
        print(f"  {str(k):30s}: {v:4d}  ({v/n*100:.1f}%)")

dist_summary(c_feat_match,       'Cancer')
dist_summary(matched_normal_list, 'Matched Normals')


Cancer  (n=1486)

── BIRADS Combo ──────────────────────────────
  4              :  565  (38.0%)
  2+4            :  225  (15.1%)
  0+0            :  146  (9.8%)
  5              :  142  (9.6%)
  4+4            :  138  (9.3%)
  2+5            :  126  (8.5%)
  5+5            :   62  (4.2%)
  2+2            :   26  (1.7%)
  1+4            :   21  (1.4%)
  1+5            :   14  (0.9%)
  3+5            :    5  (0.3%)
  4+5            :    4  (0.3%)
  1+1            :    4  (0.3%)
  3+4            :    4  (0.3%)
  0+5            :    2  (0.1%)
  3+3            :    1  (0.1%)
  0+4            :    1  (0.1%)

── Density ───────────────────────────────────
  C              : 1455  (97.9%)
  D              :   31  (2.1%)

── Age Bin ───────────────────────────────────
  40-44          :  108  (7.3%)
  45-49          :  138  (9.3%)
  50-54          :  160  (10.8%)
  55-59          :  210  (14.1%)
  60-64          :  221  (14.9%)
  65-69          :  234  (15.7%)
  70-74          :  201  (13.5%

In [26]:
matched_normal_list

,PATIENT_STUDY_ID,birads_combo,density,PatientAge,age_bin,race,stratum,matched_cancer_id
0,4334383907,2+2,C,50.0,50-54,ASIAN,C|ASIAN|50-54,4330018595
1,4333103448,1+1,C,50.0,50-54,ASIAN,C|ASIAN|50-54,4330018595
2,4334755941,2+2,C,54.0,50-54,ASIAN,C|ASIAN|50-54,4330018595
3,4335075960,2+2,C,54.0,50-54,ASIAN,C|ASIAN|50-54,4330018595
4,4334303694,2+2,C,79.0,75-79,WHITE,C|WHITE|75-79,4330303540
...,...,...,...,...,...,...,...,...
5928,4333958367,2+2,C,67.0,65-69,WHITE,C|WHITE|65-69,4339604670
5929,4333728979,2+2,C,41.0,40-44,WHITE,C|WHITE|40-44,4339617268
5930,4333062316,2+2,C,43.0,40-44,WHITE,C|WHITE|40-44,4339617268
5931,4334424223,2+2,C,44.0,40-44,WHITE,C|WHITE|40-44,4339617268


In [27]:
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "matched_correspondance_from_cancer_candidate" + ".xlsx")
matched_normal_list.to_excel(file_path, index=False)

In [28]:
matched_normal_candidate_ehr = normal_candidate_ehr[
    normal_candidate_ehr['PATIENT_STUDY_ID'].isin(matched_normal_list['PATIENT_STUDY_ID'])
].copy()

print(f"Matched normal EHR rows : {len(matched_normal_candidate_ehr)}")
print(f"Unique patients         : {matched_normal_candidate_ehr['PATIENT_STUDY_ID'].nunique()}")

Matched normal EHR rows : 5933
Unique patients         : 5933


In [29]:
matched_normal_candidate = normal_candidate[
    normal_candidate['PATIENT_STUDY_ID'].isin(matched_normal_list['PATIENT_STUDY_ID'])
].copy()

print(f"Matched normal EHR rows : {len(matched_normal_candidate)}")
print(f"Unique patients         : {matched_normal_candidate['PATIENT_STUDY_ID'].nunique()}")

Matched normal EHR rows : 23105
Unique patients         : 5933


In [30]:
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "matched_normal_candidate_ehr_from_cancer_candidate" + ".xlsx")
matched_normal_candidate_ehr.to_excel(file_path, index=False)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "matched_normal_candidate_from_cancer_candidate" + ".xlsx")
matched_normal_candidate.to_excel(file_path, index=False)